In [2]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PromptTuningConfig, PromptTuningInit, TaskType, get_peft_model

### Load SAMSum dataset

In [3]:
samsum = load_dataset("knkarthick/samsum")
print(samsum)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


_Make smaller splits_

In [4]:
train_raw = samsum["train"].shuffle(seed=42).select(range(3000))
val_raw = samsum["validation"].select(range(300))
test_raw = samsum["test"].select(range(300))

print("Train:", len(train_raw))
print("Val  :", len(val_raw))
print("Test :", len(test_raw))

Train: 3000
Val  : 300
Test : 300


_Show one sample_

In [5]:
def build_source_text(dialogue):
    return "Summarize this dialogue:\n" + dialogue.strip()

sample = train_raw[0]

print("Dialogue:\n")
print(sample["dialogue"])
print("\nReal Summary:\n")
print(sample["summary"])
print("\nDiscrete Prompt Text:\n")
print(build_source_text(sample["dialogue"]))

Dialogue:

Adam: <file_video>
Adam: what do you think
Hector: give me a sec
Hector: ok watching
Adam: let me know
Hector: can't really hear a lot there ;/
Adam: yeah ;/
Adam: i think i need to record it somehow else
Adam: maybe through the interface and software
Hector: that definitely is a great idea!
Hector: i guess that's why i gave you the interface and installed it :D
Adam: yeah xd
Adam: ok i'll try to figure it out later
Hector: ok
Hector: i'll be waiting :P

Real Summary:

Adam will record it somewhere else through the interface and software. Hector gave and installed the interface before.

Discrete Prompt Text:

Summarize this dialogue:
Adam: <file_video>
Adam: what do you think
Hector: give me a sec
Hector: ok watching
Adam: let me know
Hector: can't really hear a lot there ;/
Adam: yeah ;/
Adam: i think i need to record it somehow else
Adam: maybe through the interface and software
Hector: that definitely is a great idea!
Hector: i guess that's why i gave you the interface an

### Load tokenizer

In [6]:
MODEL_NAME = "facebook/bart-base"
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer.pad_token)
print(tokenizer.eos_token)

<pad>
</s>


### Tokenize each example

In [7]:
def preprocess(example):
    source_text = build_source_text(example["dialogue"])
    target_text = example["summary"].strip()
    
    model_inputs = tokenizer(
        source_text,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH
    )
    
    labels = tokenizer(
        text_target=target_text,
        truncation=True,
        max_length=MAX_TARGET_LENGTH
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

_Tokenize datasets_

In [8]:
train_dataset = train_raw.map(preprocess, remove_columns=train_raw.column_names)
val_dataset = val_raw.map(preprocess, remove_columns=val_raw.column_names)
test_dataset = test_raw.map(preprocess, remove_columns=test_raw.column_names)

print(train_dataset)
print(train_dataset[0].keys())

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [9]:
print("Input length :", len(train_dataset[0]["input_ids"]))
print("Label length :", len(train_dataset[0]["labels"]))

Input length : 146
Label length : 22


### Build collate function and dataloaders

In [10]:
def collate_fn(batch):
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"], dtype=torch.long) for item in batch]
    labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]
    
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

_Dataloaders_

In [11]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

_Inspect one batch_

In [ ]:
batch = next(iter(train_loader))

print("input_ids shape :", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels shape :", batch["labels"].shape)
print("First label row :", batch["labels"][0][:30])

input_ids shape     : torch.Size([8, 303])
attention_mask shape: torch.Size([8, 303])
labels shape        : torch.Size([8, 64])
First label row     : tensor([    0, 19860,  6897,     7,   907,   103, 30679,  1790,     8,  3984,
           13,  3309,   102,    15,    39,   169,   184,     4,     2,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100])
